# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ahmed-Zia/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule: "Visible & Under-Clicked" — flag pages that already earn
meaningful search visibility (real impressions) but under-perform
their position tier's expected CTR. These are pages that don't need
new content or a ranking push — they need a metadata/title/snippet
fix, because the visibility is already there and the click isn't
converting the way pages at that position tier normally do.

Score: gsc_impressions * max(0, tier_avg_ctr - page_ctr)
  — rewards pages with BOTH real volume AND a real CTR shortfall
  relative to peers at the same position; a page with no shortfall
  or no volume scores near zero either way.

Reason code (ONE): ctr_underperforms_position_tier
Action label (ONE): review_metadata_snippet

Signals this leans on, both tied to real FlyRank flags named in the
session:
1. CTR-vs-position (behind the CTR-fix logic)
2. Volume/impressions (behind the quick-win flag)

In [7]:
%pip install -q duckdb huggingface_hub
import duckdb, pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

# Pull month-level aggregates per page, real GSC columns only
base = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{REL}/**/fact_content_daily_performance/**/month={MONTH}/*.parquet')
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()
base["ctr"] = base["clicks"] / base["impressions"]

# Position tiers, same buckets the lane guide uses
def tier(p):
    if p <= 3: return "top_3"
    elif p <= 10: return "page_1"
    elif p <= 20: return "striking"
    elif p <= 50: return "page_3_5"
    else: return "deep"
base["position_tier"] = base["avg_position"].apply(tier)

# SIGNAL CHECK 1 — CTR vs position tier (behind CTR-fix logic)
signal1 = base.groupby("position_tier").agg(
    mean_ctr=("ctr", "mean"), n=("ctr", "count")
).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("Signal 1 — CTR by position tier:")
print(signal1)
print("\nVerdict: CONFIRMED — CTR drops in a clean, near-monotonic step from top_3 (1.24%) down to deep (0.09%) across n=11,681 to n=81,988 per tier. Position tier is a real, strong predictor of expected click behavior, exactly as the CTR-fix logic assumes.")

# SIGNAL CHECK 2 — volume vs position (behind quick-win logic)
base["volume_tier"] = pd.qcut(base["impressions"], 4, labels=["low", "medium", "high", "very_high"])
signal2 = base.groupby("volume_tier").agg(
    mean_position=("avg_position", "mean"), n=("avg_position", "count")
)
print("\nSignal 2 — avg_position by volume tier:")
print(signal2)
print("\nVerdict: MIXED — avg_position does NOT improve monotonically with volume. Low-volume pages average position 15.38, but medium-volume pages are actually WORSE at 22.03, before improving again at high (15.69) and very_high (11.01) — each bucket ~44,000 pages. Volume alone is not a clean predictor of position; only the top volume quartile shows a real advantage. This is exactly why my rule uses volume only as a multiplier alongside a real CTR gap, never as a standalone signal — Signal 2 just saved the rule from over-trusting volume on its own.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 1 — CTR by position tier:
               mean_ctr      n
position_tier                 
top_3          0.012399  17578
page_1         0.004926  81988
striking       0.003211  32203
page_3_5       0.002287  33288
deep           0.000903  11681

Verdict: CONFIRMED — CTR drops in a clean, near-monotonic step from top_3 (1.24%) down to deep (0.09%) across n=11,681 to n=81,988 per tier. Position tier is a real, strong predictor of expected click behavior, exactly as the CTR-fix logic assumes.

Signal 2 — avg_position by volume tier:
             mean_position      n
volume_tier                      
low              15.378710  44983
medium           22.032410  43409
high             15.692132  44186
very_high        11.008204  44160

Verdict: MIXED — avg_position does NOT improve monotonically with volume. Low-volume pages average position 15.38, but medium-volume pages are actually WORSE at 22.03, before improving again at high (15.69) and very_high (11.01) — each bucket ~44,000 pag

/tmp/ipykernel_456/2033599025.py:43: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal2 = base.groupby("volume_tier").agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = impressions * max(0, tier_avg_ctr - page_ctr). Reason code
and action label are constant (this is one rule, not a multi-branch
system) — every flagged page gets the same explanation because
they're all being flagged for the same reason.

In [8]:
import os

tier_avg_ctr = base.groupby("position_tier")["ctr"].transform("mean")
base["ctr_gap"] = (tier_avg_ctr - base["ctr"]).clip(lower=0)
base["score"] = base["impressions"] * base["ctr_gap"]

base["reason_code"] = "ctr_underperforms_position_tier"
base["action"] = "review_metadata_snippet"

queue = base.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue[["content_hash_id", "position_tier", "impressions", "clicks", "ctr",
       "avg_position", "score", "reason_code", "action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)

print("Wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
print(queue.head(20)[["content_hash_id", "position_tier", "impressions", "ctr", "avg_position", "score"]])

Wrote 176738 rows to work/outputs/baseline_action_score.csv
             content_hash_id position_tier  impressions       ctr  \
0   content_8d7d99f109e19aa2         top_3     203497.0  0.001420   
1   content_0e03de7680314cd5         top_3     221310.0  0.003253   
2   content_eadb33b5df496f4a         top_3     617124.0  0.009185   
3   content_4ffe18112a5642e3         top_3     186983.0  0.003134   
4   content_ec2e0346994fb5a5         top_3     245276.0  0.006034   
5   content_545bb6cc7081ded3         top_3     122905.0  0.002335   
6   content_44f34c0a90047651        page_1     212404.0  0.000113   
7   content_9ef3d7516483e665         top_3      89229.0  0.001031   
8   content_306bc78dff1eb683         top_3      80821.0  0.000433   
9   content_987d251ee617d9c6         top_3     152806.0  0.006152   
10  content_c46df0fa61530d86         top_3      70398.0  0.000597   
11  content_80eb6221de550658         top_3      79766.0  0.002194   
12  content_e0ca055423cbe896         top_3 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
top20 = queue.head(20)[["content_hash_id", "position_tier", "impressions", "ctr", "avg_position", "score"]]
print(top20.to_string())


             content_hash_id position_tier  impressions       ctr  avg_position        score
0   content_8d7d99f109e19aa2         top_3     203497.0  0.001420      2.563756  2234.248399
1   content_0e03de7680314cd5         top_3     221310.0  0.003253      2.675217  2024.119585
2   content_eadb33b5df496f4a         top_3     617124.0  0.009185      2.383011  1983.990668
3   content_4ffe18112a5642e3         top_3     186983.0  0.003134      2.331060  1732.484083
4   content_ec2e0346994fb5a5         top_3     245276.0  0.006034      2.854514  1561.284512
5   content_545bb6cc7081ded3         top_3     122905.0  0.002335      2.615390  1236.952906
6   content_44f34c0a90047651        page_1     212404.0  0.000113      7.346909  1022.313231
7   content_9ef3d7516483e665         top_3      89229.0  0.001031      2.481596  1014.389438
8   content_306bc78dff1eb683         top_3      80821.0  0.000433      1.488604   967.134964
9   content_987d251ee617d9c6         top_3     152806.0  0.006152     

1. content_8d7d99f109e19aa2 — top_3, pos 2.56, 203,497 impr,
   CTR 0.14% (vs 1.24% tier avg) | Flagged for the largest absolute
   CTR shortfall in the queue | Would be wrong if this is a branded
   query where users already recognize the result and skip to a
   sitelink or maps pack instead of clicking the main URL.

2. content_0e03de7680314cd5 — top_3, pos 2.68, 221,310 impr,
   CTR 0.33% | Large volume, CTR ~74% below tier average | Would be
   wrong if this page recently changed position and CTR hasn't
   caught up yet — a lag artifact, not a real snippet problem.

3. content_eadb33b5df496f4a — top_3, pos 2.38, 617,124 impr,
   CTR 0.92% | Highest volume in the queue; even a moderate relative
   gap (26% below tier avg) produces a huge score here | Would be
   wrong if this page's CTR is already close to what's realistic for
   its actual query mix — the gap is smaller than most others in the
   top 20, so this may be lower-priority than its rank suggests.

4. content_4ffe18112a5642e3 — top_3, pos 2.33, 186,983 impr,
   CTR 0.31% | Strong position, CTR ~75% below tier average | Would
   be wrong if the SERP shows a rich snippet/featured answer that
   satisfies the query without a click — position alone doesn't
   guarantee a normal CTR ceiling.

5. content_ec2e0346994fb5a5 — top_3, pos 2.85, 245,276 impr,
   CTR 0.60% | Moderate gap, high volume | Would be wrong if this
   query has strong seasonal variation and this month is naturally
   low-click for unrelated reasons.

6. content_545bb6cc7081ded3 — top_3, pos 2.62, 122,905 impr,
   CTR 0.23% | Large relative gap (81% below tier avg) | Would be
   wrong if this is a duplicate/near-duplicate of another ranking
   page on the same site, splitting clicks that a single page would
   otherwise capture.

7. content_44f34c0a90047651 — page_1, pos 7.35, 212,404 impr,
   CTR 0.011% | The single lowest CTR in the entire top 20, even
   accounting for the lower page_1 tier average (0.49%) | Would be
   wrong if this is an informational/definition-style query fully
   answered in the snippet — a near-zero click rate can be normal
   for "quick answer" intent, not a snippet defect.

8. content_9ef3d7516483e665 — top_3, pos 2.48, 89,229 impr,
   CTR 0.10% | Large relative gap on moderate volume | Would be
   wrong for the same reasons as row 1 — worth checking if this is
   a branded result.

9. content_306bc78dff1eb683 — top_3, pos 1.49 (effectively #1),
   80,821 impr, CTR 0.04% | Ranking #1 with almost no clicks — this
   is the sharpest anomaly in the whole queue | Would be wrong if
   this is a metadata/snippet issue at all — position this strong
   with CTR this low more likely means cannibalization (another URL
   capturing the real clicks), a featured snippet stealing the
   click, or a tracking bug. I'd manually inspect this one before
   trusting the "review_metadata_snippet" action label.

10. content_987d251ee617d9c6 — top_3, pos 2.82, 152,806 impr,
    CTR 0.62% | Moderate gap | Would be wrong if this CTR is already
    reasonable for a lower-intent/navigational query type.

11. content_c46df0fa61530d86 — top_3, pos 1.56 (effectively #1),
    70,398 impr, CTR 0.06% | Same #1-position-near-zero-CTR pattern
    as row 9 | Would be wrong for the same reason — likely
    cannibalization or a snippet-capture issue, not a title problem.

12. content_80eb6221de550658 — top_3, pos 2.23, 79,766 impr,
    CTR 0.22% | Large relative gap, smaller volume | Would be wrong
    if impressions are inflated by a broad-match query variant that
    doesn't reflect the page's real audience.

13. content_e0ca055423cbe896 — top_3, pos 2.68, 86,319 impr,
    CTR 0.31% | Moderate gap | Same branded-query caveat as row 1.

14. content_b13e95d379c78818 — top_3, pos 1.14 (essentially #1),
    76,121 impr, CTR 0.20% | Third instance of the #1-position/
    near-zero-CTR pattern | Same caveat as rows 9 and 11 — this
    cluster deserves lower trust in the action label as written.

15. content_6a9c79f55413b447 — top_3, pos 2.55, 73,272 impr,
    CTR 0.16% | Large relative gap on modest volume | Would be wrong
    if this page is new and hasn't accumulated a stable CTR yet —
    check content_age_days before trusting this score.

16. content_252aa5480bb1f8d7 — top_3, pos 2.39, 66,698 impr,
    CTR 0.11% | Similar pattern to row 15 | Same new-page caveat.

17. content_b2b85c287474668d — top_3, pos 1.54 (effectively #1),
    65,304 impr, CTR 0.09% | Fourth instance of the #1-position
    anomaly cluster | Same cannibalization/snippet-capture caveat.

18. content_fc67675904376267 — top_3, pos 2.26, 60,172 impr,
    CTR 0.03% | Near-zero CTR at a strong (non-#1) position | Would
    be wrong if this is a tracking gap rather than a real click
    shortfall — worth confirming the impression count itself is real
    and not a bot/crawler artifact.

19. content_60b99970e55b1ac5 — top_3, pos 2.94, 65,995 impr,
    CTR 0.17% | Weakest-volume entry in the top 20, near the tail
    of what still clears a meaningful score | Lower confidence pick.

20. content_6b4d5458bf2143b1 — top_3, pos 2.85, 67,642 impr,
    CTR 0.24% | Smallest score in the top 20 | Lower confidence —
    this is closer to the noise floor of the rule than a genuinely
    strong signal.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weakest picks, by evidence: two distinct issues, not one.

(1) Structural bias toward top_3: the rule's ceiling is tier_avg_ctr,
and top_3 has the highest ceiling (1.24%) of any tier — so top_3
pages can rack up the largest absolute gaps almost by construction,
which is why 19 of 20 top rows are top_3. This doesn't mean issues
in page_1 or striking tiers are less real, just that they can't
mathematically out-score a top_3 page with comparable volume. Worth
flagging as a known bias in this baseline, not a bug — a future
model-based version should probably normalize by tier ceiling instead
of raw CTR gap.

(2) The #1-position/near-zero-CTR cluster (rows 9, 11, 14, 17): four
pages rank at position ~1.1-1.6 yet get almost no clicks. That's
unusual enough that "review_metadata_snippet" is probably the wrong
action label for these specific four — cannibalization, a featured
snippet stealing the click, or a tracking issue are all more likely
explanations than a bad title/meta description at position #1. These
four get a manual-review flag rather than trusted at face value.

In [10]:
used_cols = ["content_hash_id", "impressions", "clicks", "ctr", "avg_position", "position_tier"]
forbidden = ["health_score", "priority_score", "action_type", "refresh_tier", "trend_pct", "trend_direction"]
leaked = [c for c in used_cols if c in forbidden]
print("Feature columns actually used:", used_cols)
print("Any forbidden/product-flag or label-derived columns present:", leaked if leaked else "none — clean")
print("Time window used: single month (2026-03), no future window referenced.")


Feature columns actually used: ['content_hash_id', 'impressions', 'clicks', 'ctr', 'avg_position', 'position_tier']
Any forbidden/product-flag or label-derived columns present: none — clean
Time window used: single month (2026-03), no future window referenced.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.